# Task 4 - Game Title Detection from Screenshots (5 classes, metric: F1 macro)

CPE342 Karena Player Intelligence System

> **Rules followed:** image classification may use pre-trained weights (ImageNet-pretrained ConvNeXt-Small fine-tuned on the task images, no AutoML); every statistic (class weights, fold split, best-epoch choice) is computed from the **train** set only; the test set is used only for the final prediction.
>
> Needs `torch`, `torchvision`, `timm`, `pillow`; a GPU is strongly recommended (`QUICK=1` runs a tiny smoke test).

## 1. Import Libraries & Configuration

In [ ]:
import os, random, time, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms as T
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings("ignore")

SEED = 42
QUICK = os.environ.get("QUICK") == "1"   # QUICK=1 -> small smoke test (results not meaningful)
ROOT = Path.cwd()                         # repo root, or the task4/ folder itself
OUT = ROOT / ("outputs_quick" if QUICK else "outputs")
(OUT / "models").mkdir(parents=True, exist_ok=True)
print("QUICK mode:", QUICK)

TASK = 4
NAMES = ["7-11 Knights", "FiveN", "Free Fried", "PUBE", "Roblock"]
DATA = ROOT / "task4" if (ROOT / "task4" / "train.csv").exists() else ROOT   # run from repo root or from inside task4/
MODEL_NAME = "convnext_small.fb_in22k_ft_in1k"

CFG = {
    "n_folds": 5,
    "train_folds": [0],                    # fold 0 of stratified 5-fold (matches best-scoring Kaggle submission)
    "img_size": (128, 256),                # preserve landscape aspect ratio of 230x120 screenshots
    "batch_size": 32 if QUICK else 64,
    "epochs": 1 if QUICK else 8,
    "lr": 2e-4,
    "weight_decay": 1e-2,
    "num_classes": 5,
    "num_workers": 0 if QUICK else 2,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
print("device:", CFG["device"])

def seed_everything(seed=SEED):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
seed_everything()

In [ ]:
# ---------- helper functions (all statistics are fitted on TRAIN only) ----------
NORM = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_transform = T.Compose([
    T.Resize(CFG["img_size"]),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    T.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.9, 1.05)),
    T.ToTensor(), T.Normalize(**NORM),
    T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])
val_transform = T.Compose([T.Resize(CFG["img_size"]), T.ToTensor(), T.Normalize(**NORM)])

def img_filenames(df):
    """Use `file_name` column from CSV when present, falling back to `<id>.jpg`."""
    if "file_name" in df.columns:
        return df["file_name"].astype(str).values
    return np.array([f"{i}.jpg" for i in df["id"].astype(str).values])

class GameDataset(Dataset):
    """Screenshot dataset: files are `<img_dir>/<file_name>` (or `<id>.jpg`); labels are optional (test)."""
    def __init__(self, df, img_dir, transform=None, target=None):
        self.files, self.img_dir, self.transform = img_filenames(df), img_dir, transform
        self.y = df[target].astype(int).values if target else None
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.img_dir / self.files[i]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return (img, int(self.y[i])) if self.y is not None else (img, 0)

def make_loader(df, img_dir, transform, shuffle, target=None):
    return DataLoader(GameDataset(df, img_dir, transform, target), batch_size=CFG["batch_size"] * (1 if shuffle else 2),
                      shuffle=shuffle, num_workers=CFG["num_workers"], pin_memory=CFG["device"] == "cuda", drop_last=shuffle)

@torch.no_grad()
def predict_proba(model, loader, tta=True):
    """Softmax probabilities; TTA = average of the image and its left-right flip."""
    model.eval(); out = []
    for x, _ in loader:
        x = x.to(CFG["device"])
        with torch.autocast(CFG["device"], enabled=CFG["device"] == "cuda"):
            p = torch.softmax(model(x), 1)
            if tta:
                p = (p + torch.softmax(model(torch.flip(x, dims=[3])), 1)) / 2
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)

def confusion_plot(y, pred, labels=None, title="Confusion matrix (row-normalised, out-of-fold)"):
    cm = confusion_matrix(y, pred, normalize="true")
    k = cm.shape[0]
    plt.figure(figsize=(6, 5)); plt.imshow(cm, cmap="Blues", vmin=0, vmax=1); plt.colorbar()
    for i in range(k):
        for j in range(k):
            plt.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center")
    plt.xticks(range(k), labels or range(k), rotation=30); plt.yticks(range(k), labels or range(k))
    plt.xlabel("predicted"); plt.ylabel("true"); plt.title(title); plt.tight_layout(); plt.show()

def save_outputs(task, pred, model_obj, extra=None, ids=None):
    id_col = ids if ids is not None else [f"ANS{i + 1:05d}" for i in range(len(pred))]
    df = pd.DataFrame({"id": id_col, "pred": pred})
    for k, v in (extra or {}).items():
        df[k] = v
    df.to_csv(OUT / f"task{task}_pred.csv", index=False)
    joblib.dump(model_obj, OUT / "models" / f"task{task}_model.joblib")
    print(f"saved {len(df)} predictions -> {OUT / f'task{task}_pred.csv'}")
    print(f"saved model -> {OUT / 'models' / f'task{task}_model.joblib'}")

## 2. Load Data

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
TARGET = "class" if "class" in train.columns else "label"     # datacard calls it `class`
TRAIN_IMG, TEST_IMG = DATA / "train", DATA / "test"
if QUICK:
    train = pd.concat([d.sample(min(len(d), 20), random_state=SEED) for _, d in train.groupby(TARGET)], ignore_index=True)
    test = test.head(64).reset_index(drop=True)
print(f"Train shape: {train.shape}\nTest shape:  {test.shape}\nmissing labels: {int(train[TARGET].isna().sum())}")
train.head()

## 3. Exploratory Data Analysis (EDA)

EDA uses the **train** set only.

In [ ]:
vc = train[TARGET].value_counts(normalize=True).sort_index()
print(vc.set_axis(NAMES).round(3))
vc.set_axis(NAMES).plot(kind="bar", color="steelblue", figsize=(7, 4), title="Class share (train)")
plt.xticks(rotation=30); plt.show()

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(15, 7))
for k, row in enumerate(axes):
    sub = train.loc[train[TARGET] == k].sample(5, random_state=SEED)
    for ax, fn in zip(row, img_filenames(sub)):
        ax.imshow(Image.open(TRAIN_IMG / fn).convert("RGB")); ax.axis("off")
    row[0].set_title(NAMES[k], loc="left", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# image size / colour check on a random sample (train only)
sample = train.sample(min(300, len(train)), random_state=SEED)
sample_files = img_filenames(sample)
sizes = pd.Series([Image.open(TRAIN_IMG / fn).size for fn in sample_files]).value_counts()
print("image sizes (w, h):", dict(sizes))
mean_rgb = np.array([np.asarray(Image.open(TRAIN_IMG / fn).convert("RGB")).mean((0, 1)) for fn in sample_files])
print("mean RGB per class (sample):")
print(pd.DataFrame(mean_rgb, columns=list("RGB")).groupby(sample[TARGET].values).mean().set_axis(NAMES).round(1))

## 4. Data Preprocessing

- resize every screenshot to 128x256 (preserving the 230x120 landscape aspect ratio) and normalise with the ImageNet mean / std
- augmentation (train only): horizontal flip, colour jitter, small affine, random erasing - suited to game screenshots / HUD
- stratified 5-fold split (fold ids assigned on the **train** set); square-root class weights `balanced ** 0.5` computed per fold from the training part only

In [ ]:
skf = StratifiedKFold(CFG["n_folds"], shuffle=True, random_state=SEED)
train["fold"] = -1
for fold, (_, va) in enumerate(skf.split(train, train[TARGET])):
    train.loc[train.index[va], "fold"] = fold
print(train.groupby("fold")[TARGET].value_counts().unstack().set_axis(range(CFG["n_folds"])))

## 5. Model Training & Validation

### 5.1 Fine-tune ConvNeXt-Small with stratified K-fold
`convnext_small.fb_in22k_ft_in1k` (timm) fine-tuned end-to-end at 128x256: AdamW (`lr=2e-4`, `weight_decay=1e-2`) + cosine schedule, label smoothing 0.05, mixed precision, square-root class-weighted cross-entropy (`balanced ** 0.5`).
The best epoch of each fold (by validation macro-F1) is kept; test probabilities are predicted with horizontal-flip TTA (and averaged over folds if multiple folds are run).

In [ ]:
def run_fold(fold):
    trn, val = train[train.fold != fold].reset_index(drop=True), train[train.fold == fold]
    val_idx, val = val.index.values, val.reset_index(drop=True)
    cw = compute_class_weight("balanced", classes=np.arange(CFG["num_classes"]), y=trn[TARGET].astype(int))
    w = torch.tensor(np.power(cw, 0.5), dtype=torch.float, device=CFG["device"])
    tr_loader = make_loader(trn, TRAIN_IMG, train_transform, True, TARGET)
    va_loader = make_loader(val, TRAIN_IMG, val_transform, False, TARGET)

    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=CFG["num_classes"]).to(CFG["device"])
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"] * len(tr_loader), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=CFG["device"] == "cuda")

    best_f1, best_state, best_val = -1, None, None
    for ep in range(CFG["epochs"]):
        model.train(); loss_sum = 0.0; t0 = time.time()
        for x, y in tr_loader:
            x, y = x.to(CFG["device"]), y.to(CFG["device"])
            opt.zero_grad()
            with torch.autocast(CFG["device"], enabled=CFG["device"] == "cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            loss_sum += loss.item() * len(x)
        pv = predict_proba(model, va_loader, tta=False)
        f1 = f1_score(val[TARGET].astype(int), pv.argmax(1), average="macro")
        print(f"fold {fold} | epoch {ep + 1:02d}/{CFG['epochs']} | train loss {loss_sum / (len(tr_loader) * CFG['batch_size']):.4f} | val macro-F1 {f1:.5f} | {time.time() - t0:4.0f}s")
        if f1 > best_f1:
            best_f1, best_val = f1, pv
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    pt = predict_proba(model, make_loader(test, TEST_IMG, val_transform, False), tta=True)
    print(f"fold {fold} best macro-F1: {best_f1:.5f}")
    return val_idx, best_val, pt, best_state

### 5.2 Run the folds and evaluate out-of-fold

In [ ]:
oof = np.full((len(train), CFG["num_classes"]), np.nan)
pt = np.zeros((len(test), CFG["num_classes"]))
fold_states = []
for fold in CFG["train_folds"]:
    seed_everything(SEED + fold)
    vi, pv, pte, state = run_fold(fold)
    oof[vi] = pv
    pt += pte / len(CFG["train_folds"])
    fold_states.append(state)

done = ~np.isnan(oof[:, 0])
y = train[TARGET].astype(int).values
print(f"\nOOF macro-F1 on {int(done.sum())} validated rows ({len(CFG['train_folds'])}/{CFG['n_folds']} folds): "
      f"{f1_score(y[done], oof[done].argmax(1), average='macro'):.4f}")
print(classification_report(y[done], oof[done].argmax(1), labels=range(CFG["num_classes"]), target_names=NAMES, digits=4, zero_division=0))
confusion_plot(y[done], oof[done].argmax(1), NAMES)

Reference run on Colab T4 (`convnext_small.fb_in22k_ft_in1k`, 128x256, fold 0, 8 epochs): best-epoch macro-F1 = 0.99010 (epoch 7/8); predicted test class shares = 16.28% / 23.43% / 15.77% / 26.89% / 17.63%.

## 6. Predict on the Test Set & Save

In [ ]:
pred = pt.argmax(1)
share = pd.DataFrame({
    "train": pd.Series(y).value_counts(normalize=True).reindex(range(CFG["num_classes"]), fill_value=0),
    "test predicted": pd.Series(pred).value_counts(normalize=True).reindex(range(CFG["num_classes"]), fill_value=0),
}).set_axis(NAMES)
print(share.round(3))
share.plot(kind="bar", figsize=(7, 4), color=["steelblue", "tomato"], title="Class share: train vs predicted test")
plt.xticks(rotation=30); plt.show()
save_outputs(TASK, pred, {"name": MODEL_NAME, "fold_state_dicts": fold_states, "cfg": CFG, "classes": NAMES},
             {f"p{k}": pt[:, k] for k in range(CFG["num_classes"])}, ids=test["id"].values)